<a href="https://colab.research.google.com/github/UnitedDobermanRescue/UDR/blob/main/UDR_foster_map.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import pandas as pd
import folium
from folium.plugins import MarkerCluster
import json


In [7]:
# Adjust path as needed
df = pd.read_csv("animals.csv")


In [8]:
df_fostered = df[
    df["City"].notna() &
    (df["City"] != "") &
    df["State"].notna() &
    (df["State"] != "")
].reset_index(drop=True)

df_needs_foster = df[
    ((df["City"].isna()) | (df["City"] == "")) &
    ((df["State"].isna()) | (df["State"] == ""))
].reset_index(drop=True)


### Geocoding City and State to Latitude and Longitude

To display the dogs on a map, we need their precise geographical coordinates (latitude and longitude). We will use the `geopy` library to convert the 'City' and 'State' information in our `df_fostered` DataFrame into 'lat' and 'lon' columns. This process is called geocoding.

In [9]:
%%capture
pip install geopy

In [10]:
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

geolocator = Nominatim(user_agent="my_geocoder_app")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

def get_coordinates(city, state):
    try:
        location = geocode(f"{city}, {state}")
        if location:
            return location.latitude, location.longitude
        else:
            return None, None
    except Exception as e:
        print(f"Error geocoding {city}, {state}: {e}")
        return None, None

# Apply geocoding to df_fostered
df_fostered[['lat', 'lon']] = df_fostered.apply(
    lambda row: get_coordinates(row['City'], row['State']),
    axis=1,
    result_type='expand'
)

# Drop rows where geocoding failed (lat/lon are None)
df_fostered.dropna(subset=['lat', 'lon'], inplace=True)

display(df_fostered.head())


,Name,Status,Species,Sex,Days in Foster,City,Name.1,State,Zip/Postal Code,Animal ID,...,Reaction to New People,Requires a Home with Fence,Requires a Yard,Secondary Breed,Special Needs (description),Summary,Thumbnail,Video URL 1,lat,lon
0,Jessie,Available,Dog,Female,NaN,Racine,Volk Matthew,WI,53403.0,22583108,...,Friendly,Any Type,Yes,NaN,NaN,NaN,,NaN,42.731376,-87.783477
1,Bindi,Available,Dog,Female,NaN,Racine,Volk Matthew,WI,53403.0,22688203,...,Friendly,Any Type,Yes,NaN,NaN,NaN,,NaN,42.731376,-87.783477
2,Diamond,Available,Dog,Female,5.0,Glendale,Ruud Mary,AZ,85310.0,22694555,...,Friendly,Not Required,Yes,NaN,NaN,NaN,,NaN,33.538686,-112.185994
3,Atlas aka Doby,Available,Dog,Male,54.0,Verona,Harris Darrien,ND,58490.0,22570368,...,Cautious,6 foot,Yes,NaN,NaN,NaN,,NaN,46.363857,-98.072323
4,Bo Adoption Pending!,Available,Dog,Male,74.0,WICHITA,Kahl Paul,KS,67230.0,22511594,...,Friendly,Any Type,Yes,NaN,NaN,NaN,,NaN,37.692236,-97.337545


### Interface Component Definitions
This cell defines the HTML, CSS, and structural elements for the map sidebar, search bar, and legend.

In [11]:
sidebar_html = """
<meta name="viewport" content="width=device-width, initial-scale=1.0, maximum-scale=1.0, user-scalable=no" />
<style>
#sidebar {
    position: fixed; top: 0; right: -420px; width: 420px; height: 100%;
    background: white; border-left: 3px solid #444; overflow-y: auto;
    transition: right 0.3s ease; z-index: 999999; font-family: Arial, sans-serif;
}
#sidebar.open { right: 0; }
#sidebar-close { cursor: pointer; font-size: 28px; font-weight: bold; float: right; margin: 15px; padding: 10px 20px; }
.tab-container { margin-top: 65px; }
.tab-buttons { display: flex; border-bottom: 2px solid #ccc; }
.tab-buttons button { flex: 1; padding: 18px; background: #eee; border: none; cursor: pointer; font-weight: bold; font-size: 16px; }
.tab-buttons button.active { background: #ddd; border-bottom: 3px solid #444; }
.tab-content { display: none; padding: 15px; }
.tab-content.active { display: block; }
#sidebar img { width: 100%; border-radius: 6px; margin-bottom: 10px; }
#page-header {
    position: fixed; top: 0; left: 0; width: 100%; background: rgba(255, 255, 255, 0.95);
    display: flex; align-items: center; justify-content: center; padding: 10px 0; z-index: 9999; border-bottom: 2px solid #444;
}
#page-header img { height: 35px; margin-right: 15px; }
#header-title { font-size: 20px; font-weight: bold; font-family: Arial, sans-serif; }
</style>
<div id="page-header">
    <img src="https://s3.amazonaws.com/imagesroot.rescuegroups.org/webpages/s8319n8fohztfmoo.jpg" alt="UDR Logo">
    <div id="header-title">UDR Foster Map</div>
</div>
<div id="sidebar"><span id="sidebar-close">✕</span><div class="tab-container">
<div class="tab-buttons"><button class="tab-btn active" onclick="openTab('details')">Details</button><button class="tab-btn" onclick="openTab('links')">Links</button></div>
<div id="details" class="tab-content active"><div id="photos"></div><div id="videos"></div><div id="info-text"></div></div>
<div id="links" class="tab-content"></div></div></div>
<script>
document.getElementById("sidebar-close").onclick = function() { document.getElementById("sidebar").classList.remove("open"); };
function openTab(tabName) {
    var tabs = document.getElementsByClassName("tab-content");
    var buttons = document.getElementsByClassName("tab-btn");
    for (var i = 0; i < tabs.length; i++) { tabs[i].classList.remove("active"); buttons[i].classList.remove("active"); }
    document.getElementById(tabName).classList.add("active");
    event.currentTarget.classList.add("active");
}
</script>
"""

search_html = """
<style>
#search-container { position: fixed; top: 70px; left: 10px; z-index: 999997; width: 240px; }
#dog-search { width: 100%; padding: 8px 12px; border: 2px solid #2c3e50; border-radius: 20px; outline: none; box-shadow: 0 2px 6px rgba(0,0,0,0.2); }
</style>
<div id="search-container"><input type="text" id="dog-search" placeholder="Search dogs by name..." oninput="filterDogs()"></div>
"""

legend_html = """
<style>
#map-legend { position: fixed; bottom: 35px; right: 20px; z-index: 999998; background: white; border: 1px solid #333; border-radius: 8px; width: 190px; overflow: hidden; }
#legend-header { background: #b2bec3; padding: 10px; font-weight: 600; text-align: center; cursor: pointer; border-bottom: 1px solid #999; }
#legend-content { display: none; padding: 5px 0; }
.legend-item { display: flex; align-items: center; padding: 5px 12px; font-size: 12px; }
.legend-color { width: 15px; height: 15px; margin-right: 10px; border-radius: 50%; }
</style>
<div id="map-legend">
    <div id="legend-header" onclick="toggleLegend()">Time in Foster ⓘ</div>
    <div id="legend-content">
        <div class="legend-item"><div class="legend-color" style="background: #ffcdd2; border: 2px solid #ef5350;"></div> &lt; 30 Days</div>
        <div class="legend-item"><div class="legend-color" style="background: #ef5350;"></div> 30 - 89 Days</div>
        <div class="legend-item"><div class="legend-color" style="background: #b71c1c;"></div> 90 - 179 Days</div>
        <div class="legend-item"><div class="legend-color" style="background: #000000;"></div> 180+ Days</div>
    </div>
</div>
<script>
function toggleLegend() {
    var content = document.getElementById('legend-content');
    content.style.display = (content.style.display === 'none' || content.style.display === '') ? 'block' : 'none';
}
</script>
"""

In [12]:
def foster_color(days):
    if pd.isna(days):
        return "lightred"
    if days < 30:
        return "lightred"
    if days < 90:
        return "red"
    if days < 180:
        return "darkred"
    return "black"


In [13]:
m = folium.Map(location=[39.5, -98.35], zoom_start=5)
cluster = MarkerCluster().add_to(m)


In [14]:
sidebar_html = """
<meta name="viewport" content="width=device-width, initial-scale=1.0, maximum-scale=1.0, user-scalable=no" />
<style>
#sidebar {
    position: fixed;
    top: 0;
    right: -420px;
    width: 420px;
    height: 100%;
    background: white;
    border-left: 3px solid #444;
    overflow-y: auto;
    transition: right 0.3s ease;
    z-index: 999999;
    font-family: Arial, sans-serif;
}
#sidebar.open { right: 0; }
#sidebar-close {
    cursor: pointer;
    font-size: 28px;
    font-weight: bold;
    float: right;
    margin: 15px;
    padding: 10px 20px;
}
.tab-container { margin-top: 65px; }
.tab-buttons { display: flex; border-bottom: 2px solid #ccc; }
.tab-buttons button {
    flex: 1; padding: 18px; background: #eee;
    border: none; cursor: pointer; font-weight: bold;
    font-size: 16px;
}
.tab-buttons button.active {
    background: #ddd; border-bottom: 3px solid #444;
}
.tab-content { display: none; padding: 15px; }
.tab-content.active { display: block; }
#sidebar img { width: 100%; border-radius: 6px; margin-bottom: 10px; }

#page-header {
    position: fixed;
    top: 0;
    left: 0;
    width: 100%;
    background: rgba(255, 255, 255, 0.95);
    display: flex;
    align-items: center;
    justify-content: center;
    padding: 10px 0;
    font-family: Arial, sans-serif;
    z-index: 9999;
    border-bottom: 2px solid #444;
}
#page-header img {
    height: 35px;
    margin-right: 10px;
}
#header-title {
    font-size: 20px;
    font-weight: bold;
}

/* Mobile Specific Overrides */
@media only screen and (max-width: 600px) {
    #sidebar {
        width: 90% !important;
        right: -90%;
    }
    #header-title {
        font-size: 16px;
    }
    #search-container, #needs-foster-panel {
        width: 180px !important;
    }
    .foster-item {
        padding: 15px !important; /* Larger touch target */
    }
}
</style>

<div id="page-header">
    <img src="https://s3.amazonaws.com/imagesroot.rescuegroups.org/webpages/s8319n8fohztfmoo.jpg" alt="UDR Logo">
    <div id="header-title">UDR Foster Map</div>
</div>

<div id="sidebar">
    <span id="sidebar-close">✕</span>

    <div class="tab-container">
        <div class="tab-buttons">
            <button class="tab-btn active" onclick="openTab('details')">Details</button>
            <button class="tab-btn" onclick="openTab('links')">Links</button>
        </div>

        <div id="details" class="tab-content active">
            <div id="photos"></div>
            <div id="videos"></div>
            <div id="info-text"></div>
        </div>
        <div id="links" class="tab-content"></div>
    </div>
</div>

<script>
document.getElementById("sidebar-close").onclick = function() {
    document.getElementById("sidebar").classList.remove("open");
};

function openSidebar() {
    document.getElementById("sidebar").classList.add("open");
}

function openTab(tabName) {
    var tabs = document.getElementsByClassName("tab-content");
    var buttons = document.getElementsByClassName("tab-btn");

    for (var i = 0; i < tabs.length; i++) {
        tabs[i].classList.remove("active");
        buttons[i].classList.remove("active");
    }

    document.getElementById(tabName).classList.add("active");
    if(tabName === 'details') buttons[0].classList.add("active");
    else buttons[1].classList.add("active");
}
</script>
"""

In [15]:
needs_foster_html = """
<style>
#needs-foster-panel {
    position: fixed;
    top: 115px; /* Moved down to accommodate header and search bar */
    left: 10px;
    z-index: 999998;
    background: white;
    border: 1px solid #ccc;
    border-radius: 6px;
    padding: 0;
    max-height: 50%;
    width: 240px;
    overflow: hidden;
    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
    box-shadow: 0 2px 8px rgba(0,0,0,0.15);
}
#needs-foster-header {
    background: #2c3e50;
    color: white;
    padding: 10px 15px;
    margin: 0;
    cursor: pointer;
    font-size: 14px;
    font-weight: 600;
    display: flex;
    justify-content: space-between;
    align-items: center;
    border-bottom: 1px solid #ddd;
}
#needs-foster-header:hover {
    background: #34495e;
}
#foster-list {
    display: none;
    list-style-type: none;
    padding: 0;
    margin: 0;
    max-height: 300px;
    overflow-y: auto;
}
.foster-item {
    padding: 8px 15px;
    border-bottom: 1px solid #f1f1f1;
    transition: all 0.2s ease;
    font-size: 13px;
    font-weight: 400;
    color: #444;
    cursor: pointer;
}
.foster-item:hover {
    background: #f8f9fa;
    color: #c0392b;
    padding-left: 18px;
}
</style>

<div id="needs-foster-panel">
    <h3 id="needs-foster-header" onclick="toggleFosterList()">
        Dogs Needing Foster <span style='font-size: 9px;'>▼</span>
    </h3>
    <ul id="foster-list">
"""

for list_index, row in df_needs_foster.iterrows():
    needs_foster_html += f"<li class='foster-item' onclick='window.openDogFromList({list_index})'>{row['Name']}</li>"

needs_foster_html += """
    </ul>
</div>
"""

In [16]:
marker_js_data = []
base_url = "https://www.uniteddobermanrescue.org"
rescue_groups_url = "https://rescuegroups.org/manage/animals#datatable74={%22limit%22:25,%22page%22:1,%22start%22:0,%22sortbyFieldID%22:1499,%22sortorder%22:%22asc%22,%22viewID%22:%22112%22,%22viewType%22:%22Built-in%22,%22togglethumbnails%22:%22Off%22,%22searchEnabled%22:false}&"
intakes_url = "https://rescuegroups.org/manage/animals_intakes#datatable387={%22limit%22:%2225%22,%22page%22:1,%22start%22:0,%22sortbyFieldID%22:%22%22,%22sortorder%22:%22asc%22,%22viewID%22:%22211%22,%22viewType%22:%22Built-in%22,%22searchEnabled%22:false}&"

# 1. Initialize Map and Cluster
m = folium.Map(location=[39.5, -98.35], zoom_start=5)
cluster = MarkerCluster().add_to(m)

# 2. Process Geocoded Markers
for marker_index, row in df_fostered.iterrows():
    details_html = (
        f"<h2>{row['Name']}</h2>"
        f"<b>Status:</b> {row['Status']}<br>"
        f"<b>Sex:</b> {row['Sex']}<br>"
        f"<b>Age:</b> {row['General Age']}<br>"
        f"<b>Energy:</b> {row['Energy Level']}<br>"
        f"<b>City:</b> {row['City']}<br>"
        f"<b>State:</b> {row['State']}<br>"
        f"<b>Days in Foster:</b> {row['Days in Foster']}<br><br>"
        f"<b>Description:</b><br>{row['Description']}"
    )

    photos_html = ""
    for p in ["Picture 1", "Picture 2", "Picture 3", "Picture 4"]:
        if pd.notna(row[p]) and str(row[p]).strip() != "":
            img_path = str(row[p])
            full_img_url = base_url + img_path if img_path.startswith('/') else img_path
            photos_html += f"<img src='{full_img_url}' style='width:100%; margin-bottom:10px; border-radius:5px;' />"

    videos_html = ""
    if pd.notna(row["Video URL 1"]) and str(row["Video URL 1"]).strip() != "":
        vid_path = str(row["Video URL 1"])
        full_vid_url = base_url + vid_path if vid_path.startswith('/') else vid_path
        videos_html += f"<a href='{full_vid_url}' target='_blank'>Watch Video</a><br><br>"

    links_html = (
        f"<a href='https://www.uniteddobermanrescue.org/animals/detail?AnimalID={row['Animal ID']}' target='_blank'>Foster Dog's Full Profile</a><br><br>"
        f"<a href='{rescue_groups_url}' target='_blank'>Rescue Groups - Available Dogs List</a><br><br>"
        f"<a href='{intakes_url}' target='_blank'>Rescue Groups - Intake</a>"
    )

    marker_js_data.append({"details": details_html, "photos": photos_html, "videos": videos_html, "links": links_html})

    icon_color = foster_color(row["Days in Foster"])
    folium.Marker(location=[row["lat"], row["lon"]], icon=folium.Icon(color=icon_color), tooltip=row['Name']).add_to(cluster)

# 3. Add Interface Components from Global Variables
# (Removed duplicate legend addition here as it is added in a later cell)
m.get_root().html.add_child(folium.Element(sidebar_html))
m.get_root().html.add_child(folium.Element(needs_foster_html))
m.get_root().html.add_child(folium.Element(search_html))

try:
    m.get_root().html.add_child(folium.Element(bottom_links))
except NameError:
    pass

print("Map successfully assembled without duplicate legend.")

Map successfully assembled without duplicate legend.


In [17]:
# Cell removed to clean up notebook flow

In [18]:
m.save("foster_map_v2.html")

In [19]:
import json

# Prepare Data for Dogs Needing Foster
needs_foster_data = []
for idx, row in df_needs_foster.iterrows():
    d_html = f"<h2>{row['Name']}</h2><b>Status:</b> {row.get('Status', 'N/A')}<br><b>Sex:</b> {row.get('Sex', 'N/A')}<br><b>Description:</b><br>{row.get('Description', 'No description available.')}"
    p_html = ""
    for p in ["Picture 1", "Picture 2", "Picture 3", "Picture 4"]:
        if pd.notna(row.get(p)) and str(row[p]).strip() != "":
            full_img_url = base_url + str(row[p]) if str(row[p]).startswith('/') else str(row[p])
            p_html += f"<img src='{full_img_url}' style='width:100%; margin-bottom:10px; border-radius:5px;' />"
    l_html = (
        f"<a href='https://www.uniteddobermanrescue.org/animals/detail?AnimalID={row.get('Animal ID', '')}' target='_blank'>Foster Dog's Full Profile</a><br><br>"
        f"<a href='{rescue_groups_url}' target='_blank'>Rescue Groups - Available Dogs List</a><br><br>"
        f"<a href='{intakes_url}' target='_blank'>Rescue Groups - Intake</a>"
    )
    needs_foster_data.append({"details": d_html, "photos": p_html, "videos": "", "links": l_html})

js_fostered = json.dumps(marker_js_data)
js_needs = json.dumps(needs_foster_data)

sidebar_js_template = """
<script>
    window.markerData = DATA_FOSTERED;
    window.needsData = DATA_NEEDS;

    function toggleFosterList() {
        var el = document.getElementById('foster-list');
        if (el) el.style.display = (el.style.display === 'none' || el.style.display === '') ? 'block' : 'none';
    }

    window.openDogFromList = function(index) {
        if (window.needsData && window.needsData[index]) {
            window.updateSidebar(window.needsData[index]);
        }
    };

    window.updateSidebar = function(data) {
        if (!data) return;
        document.getElementById('photos').innerHTML = data.photos || '';
        document.getElementById('videos').innerHTML = data.videos || '';
        document.getElementById('info-text').innerHTML = data.details || '';
        document.getElementById('links').innerHTML = data.links || '';
        document.getElementById('sidebar').classList.add('open');

        var tabs = document.getElementsByClassName("tab-content");
        var buttons = document.getElementsByClassName("tab-btn");
        for (var i = 0; i < tabs.length; i++) {
            tabs[i].classList.remove("active");
            buttons[i].classList.remove("active");
        }
        document.getElementById('details').classList.add('active');
        buttons[0].classList.add("active");
    };
</script>
"""

sidebar_js = sidebar_js_template.replace("DATA_FOSTERED", js_fostered).replace("DATA_NEEDS", js_needs)
m.get_root().html.add_child(folium.Element("{% raw %}" + sidebar_js + "{% endraw %}"))
m.save('foster_map_v2.html')
display(m)

In [20]:
bottom_links = """
<div style="
    position: fixed;
    bottom: 10px;
    left: 10px;
    background: white;
    padding: 8px;
    border: 1px solid grey;
    font-family: Arial, sans-serif;
    z-index: 99999;
    text-align: left;
    font-size: 11px;
    box-shadow: 0 2px 5px rgba(0,0,0,0.2);
">
    <a href=\"https://www.uniteddobermanrescue.org/animals/browse\" target=\"_blank\">
        Browse All Dogs
    </a><br>

    <a href=\"https://tinyurl.com/msffpcsa\" target=\"_blank\">
        Database Report
    </a>
</div>
"""

In [21]:
legend_html = """
<style>
#map-legend {
    position: fixed;
    bottom: 35px;
    right: 20px;
    z-index: 999998;
    background: white;
    border: 1px solid #333;
    border-radius: 8px;
    padding: 0;
    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
    box-shadow: 0 4px 15px rgba(0,0,0,0.3);
    width: 190px;
    overflow: hidden;
}
#legend-header {
    background: #b2bec3; /* Darker coordinating grey */
    color: #2d3436;
    padding: 10px;
    border-bottom: 2px solid #636e72;
    cursor: pointer;
    font-weight: 600;
    text-align: center;
    font-size: 14px;
}
#legend-header:hover {
    background: #9fa8a3;
}
#legend-content {
    display: none;
    padding: 12px;
}
.legend-item {
    display: flex;
    align-items: center;
    margin-bottom: 10px;
    font-size: 12px;
    font-weight: 500;
}
.legend-color {
    width: 20px;
    height: 20px;
    margin-right: 12px;
    border-radius: 50%;
    border: 1px solid rgba(0,0,0,0.2);
}
</style>

<div id="map-legend">
    <div id="legend-header" onclick="toggleLegend()">Time in Foster ⓘ</div>
    <div id="legend-content">
        <div class="legend-item"><div class="legend-color" style="background: #ffcdd2; border: 2px solid #ef5350;"></div> < 30 Days</div>
        <div class="legend-item"><div class="legend-color" style="background: #ef5350;"></div> 30 - 89 Days</div>
        <div class="legend-item"><div class="legend-color" style="background: #b71c1c;"></div> 90 - 179 Days</div>
        <div class="legend-item"><div class="legend-color" style="background: #000000;"></div> 180+ Days</div>
    </div>
</div>

<script>
function toggleLegend() {
    var content = document.getElementById('legend-content');
    content.style.display = (content.style.display === 'none' || content.style.display === '') ? 'block' : 'none';
}
</script>
"""

# Removed the redundant add_child call to prevent duplicate legends on the map

In [22]:
# Cell removed to clean up notebook flow

In [23]:
# Cell removed to clean up notebook flow

In [24]:
from google.colab import files

# Download the generated map file
files.download('foster_map_v2.html')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Final Deployment Steps
1. Download the `foster_map_v2.html` file using the cell below.
2. Upload it to your preferred hosting service (GitHub Pages, Google Drive, etc.).
3. Share the resulting URL with your team!

In [25]:
filter_js = """
<script>
    function filterDogs() {
        var searchTerm = document.getElementById('dog-search').value.toLowerCase();
        var map = null;
        var markerCluster = null;

        // Find the map and cluster objects globally
        for (var key in window) {
            if (window[key] instanceof L.Map) {
                map = window[key];
            }
            if (window[key] instanceof L.MarkerClusterGroup) {
                markerCluster = window[key];
            }
        }

        if (!map || !markerCluster) {
            console.warn("Map or MarkerCluster not found for filtering.");
            return;
        }

        // Clear existing markers from the cluster
        markerCluster.clearLayers();

        window.markerData.forEach(function(dog) {
            // The 'name' property is now available in `window.markerData`
            // Also lat, lon, iconColor are available.
            if (dog.name.toLowerCase().includes(searchTerm)) {
                // Recreate marker and add to cluster
                var marker = L.marker([dog.lat, dog.lon], {
                    icon: L.AwesomeMarkers.icon({
                        icon: 'paw',
                        prefix: 'fa',
                        markerColor: dog.iconColor // Use the stored color
                    }),
                    // Set tooltip (name) and animalId for event binding
                    title: dog.name,
                    animalId: dog.id
                });
                markerCluster.addLayer(marker);

                // Re-bind click event for the newly created marker
                marker.on('click', function(e) {
                    var dogId = e.target.options.animalId;
                    var data = window.markerData.find(d => String(d.id) === String(dogId));
                    if (data) window.updateSidebar(data);
                });
            }
        });

        // Note: Filtering for 'needs-foster' dogs is not part of this search bar's current scope.
    }
</script>
"""


In [26]:
import folium
from folium.plugins import MarkerCluster
import json
import pandas as pd

# 1. Re-initialize Map and Cluster
m = folium.Map(location=[39.5, -98.35], zoom_start=5)
cluster = MarkerCluster().add_to(m)

# 2. Build Data and Markers using Animal ID as a primary key
final_fostered_list = []

for i, (idx, row) in enumerate(df_fostered.iterrows()):
    animal_id = str(row['Animal ID'])
    dog_name = row['Name'] # Get the dog's name
    dog_lat = row['lat'] # Get latitude
    dog_lon = row['lon'] # Get longitude
    dog_icon_color = foster_color(row['Days in Foster']) # Get calculated color

    details_html = (f"<h2>{dog_name}</h2><b>Status:</b> {row['Status']}<br><b>Sex:</b> {row['Sex']}<br>"
                    f"<b>Age:</b> {row['General Age']} <br><b>Energy:</b> {row['Energy Level']}<br>"
                    f"<b>City:</b> {row['City']}<br><b>State:</b> {row['State']}<br>"
                    f"<b>Days in Foster:</b> {row['Days in Foster']}<br><br><b>Description:</b><br>{row['Description']}")

    # Photos removed per user request
    photos_html = ""

    # Updated Link Labels and URLs per request
    links_html = (f"<a href='https://www.uniteddobermanrescue.org/animals/detail?AnimalID={animal_id}' target='_blank'>Public Website - Full Profile</a><br><br>"
                  f"<a href='https://rescuegroups.org/manage/animals#datatable74={{%22limit%22:25,%22page%22:1,%22start%22:0,%22sortbyFieldID%22:1499,%22sortorder%22:%22asc%22,%22viewID%22:%22112%22:%22viewType%22:%22Built-in%22,%22togglethumbnails%22:%22Off%22,%22searchEnabled%22:false}}&' target='_blank'>RescueGroups - Available Dogs Report</a><br><br>"
                  f"<a href='https://rescuegroups.org/manage/animals_intakes#datatable387={{%22limit%22:%2225%22:%22page%22:1,%22start%22:0,%22sortbyFieldID%22:%22%22,%22sortorder%22:%22asc%22,%22viewID%22:%22211%22:%22viewType%22:%22Built-in%22:%22searchEnabled%22:false}}&' target='_blank'>RescueGroups - Intake Report</a>")

    final_fostered_list.append({
        "id": animal_id,
        "lat": dog_lat,
        "lon": dog_lon,
        "name": dog_name,
        "iconColor": dog_icon_color,
        "details": details_html,
        "photos": photos_html,
        "videos": "",
        "links": links_html
    })

    marker = folium.Marker(
        location=[dog_lat, dog_lon],
        icon=folium.Icon(color=dog_icon_color),
        tooltip=dog_name
    )
    marker.add_to(cluster)
    marker.options['animalId'] = animal_id

# 3. Enhanced JS Sync Engine (Auto-detects Map and Cluster)
final_js = sidebar_js_template.replace("DATA_FOSTERED", json.dumps(final_fostered_list)).replace("DATA_NEEDS", js_needs)

primary_key_sync_js = """
<script>
function applyPrimaryKeySync() {
    console.log("Searching for Map and Cluster objects...");

    var foundMap = null;
    var foundCluster = null;

    // Search global scope for Leaflet objects
    for (var key in window) {
        if (window[key] instanceof L.Map) {
            foundMap = window[key];
        }
        if (window[key] instanceof L.MarkerClusterGroup) {
            foundCluster = window[key];
        }
    }

    if (foundMap && foundCluster && foundCluster._featureGroup) {
        console.log("Map and Cluster found. Binding clicks...");
        foundCluster.eachLayer(function(marker) {
            if (marker instanceof L.Marker && marker.options.animalId) {
                marker.off('click');
                marker.on('click', function(e) {
                    var dogId = e.target.options.animalId;
                    var data = window.markerData.find(d => String(d.id) === String(dogId));
                    if (data) window.updateSidebar(data);
                });
            }
        });
        console.log("Sync engine active.");
    } else {
        console.log("Still waiting for Leaflet objects... retrying");
        setTimeout(applyPrimaryKeySync, 1000);
    }
}
setTimeout(applyPrimaryKeySync, 1500);
</script>
"""

# 4. Final Assembly
m.get_root().html._children = {}
m.get_root().html.add_child(folium.Element(sidebar_html))
m.get_root().html.add_child(folium.Element(needs_foster_html))
m.get_root().html.add_child(folium.Element(legend_html))
m.get_root().html.add_child(folium.Element(bottom_links))
m.get_root().html.add_child(folium.Element(search_html))
m.get_root().html.add_child(folium.Element(filter_js))
m.get_root().html.add_child(folium.Element('{% raw %}' + final_js + '{% endraw %}'))
m.get_root().html.add_child(folium.Element(primary_key_sync_js))

display(m)
print('Map Refreshed: Photos removed from sidebar.')


Map Refreshed: Photos removed from sidebar.


In [27]:
import pandas as pd

print("--- Marker Accuracy Verification ---")
print(f"Total Dogs in Filtered List: {len(final_fostered_list)}")
print("\nFirst 5 Dogs in the synchronized array (Marker ID -> Name):")

# Extract names from the HTML details to verify index alignment
for i, dog_data in enumerate(final_fostered_list[:5]):
    # Basic parsing to get the name between <h2> and </h2>
    name = dog_data['details'].split('<h2>')[1].split('</h2>')[0]
    print(f"Index {i}: {name}")

print("\nVerification Tip: Click a marker on the map. The sidebar should show the dog name.")
print("If you click the first marker placed (chronologically in the list), it should be:", df_fostered.iloc[0]['Name'])

--- Marker Accuracy Verification ---
Total Dogs in Filtered List: 13

First 5 Dogs in the synchronized array (Marker ID -> Name):
Index 0: Jessie
Index 1: Bindi
Index 2: Diamond
Index 3: Atlas aka Doby
Index 4: Bo Adoption Pending!

Verification Tip: Click a marker on the map. The sidebar should show the dog name.
If you click the first marker placed (chronologically in the list), it should be: Jessie


### Final Deployment
The map has been assembled with all interface components. Use the cell below to view the interactive map and download the final HTML file.

In [28]:
m.save('foster_map_final.html')
from google.colab import files
files.download('foster_map_final.html')
print('Final map saved and downloaded as foster_map_final.html')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Final map saved and downloaded as foster_map_final.html
